# Lightweight Models for SoC Estimation (Embedded BMS)

Este notebook implementa modelos de regressão leves para estimação do estado de
carga relativo (SoCrel), com foco em execução em dispositivos de baixo poder
computacional (ex.: ESP32).

Os modelos são avaliados usando:
- O mesmo dataset do estudo Dhakal (LG 18650HG2)
- O mesmo protocolo experimental (split por file_id)
- As mesmas métricas (MAE, RMSE, R²)

Os resultados serão comparados com os baselines (Random Forest e ANN).


In [1]:
# CÉLULA 2 — Imports e métricas

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✔ Imports carregados")


✔ Imports carregados


In [2]:
# CÉLULA 3 — Carregamento explícito do dataset Dhakal

import pandas as pd
from pathlib import Path

data_path = Path("/kaggle/input/dhakal-full/df_prepared.pkl")

print("A verificar caminho:", data_path)

if not data_path.exists():
    raise FileNotFoundError(
        "df_prepared.pkl não encontrado.\n"
        "Confirma que o Input 'dhakal-full' está ativo."
    )

df = pd.read_pickle(data_path)

print("✔ Dataset carregado com sucesso")
print("Shape:", df.shape)
print("Colunas:", list(df.columns))
df.head(3)


A verificar caminho: /kaggle/input/dhakal-full/df_prepared.pkl
✔ Dataset carregado com sucesso
Shape: (4954914, 20)
Colunas: ['time_stamp', 'step', 'status', 'prog_time', 'step_time', 'cycle', 'cycle_level', 'procedure', 'voltage', 'current', 'temperature', 'capacity', 'whaccu', 'cnt', 'unnamed_14', 'file_id', 'temperature_folder', 'test_type', 'time_s', 'soc_rel']


,time_stamp,step,status,prog_time,step_time,cycle,cycle_level,procedure,voltage,current,temperature,capacity,whaccu,cnt,unnamed_14,file_id,temperature_folder,test_type,time_s,soc_rel
0,2018-10-26 09:29:22,22.0,DCH,29:20:19.688,00:01:00.002,0.0,0.0,LG_HG2_NN_Char,4.17604,-0.15069,23.976151,-0.00254,-0.01060,13.00000,NaN,549_C20DisCh,25degC,Cycle,0.0,0.935729
1,2018-10-26 09:30:22,22.0,DCH,29:21:19.688,00:02:00.002,0.0,0.0,LG_HG2_NN_Char,4.17301,-0.15325,23.976151,-0.00507,-0.02116,13.00000,NaN,549_C20DisCh,25degC,Cycle,60.0,0.934869
2,2018-10-26 09:31:22,22.0,DCH,29:22:19.688,00:03:00.002,0.0,0.0,LG_HG2_NN_Char,4.17014,-0.15069,23.976151,-0.00761,-0.03176,13.00000,NaN,549_C20DisCh,25degC,Cycle,120.0,0.934023


In [3]:
# CÉLULA 4 — Identificação das colunas essenciais

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_file = pick_col(df, ["file_id"])
col_time = pick_col(df, ["time_s"])
col_v    = pick_col(df, ["voltage"])
col_i    = pick_col(df, ["current"])
col_t    = pick_col(df, ["temperature"])
col_soc  = pick_col(df, ["soc_rel"])

if None in [col_file, col_time, col_v, col_i, col_t, col_soc]:
    raise ValueError("Colunas essenciais em falta.")

print("✔ Colunas usadas:")
print(col_file, col_time, col_v, col_i, col_t, col_soc)


✔ Colunas usadas:
file_id time_s voltage current temperature soc_rel


In [4]:
# CÉLULA 5 — Seleção das variáveis e ordenação temporal

use_cols = [col_file, col_time, col_v, col_i, col_t, col_soc]
dfm = df[use_cols].copy()

for c in use_cols[1:]:
    dfm[c] = pd.to_numeric(dfm[c], errors="coerce")
 
dfm = dfm.dropna()
dfm = dfm.sort_values([col_file, col_time]).reset_index(drop=True)

print("✔ Dataset limpo:", dfm.shape)
dfm.head(3)


✔ Dataset limpo: (4954914, 6)


,file_id,time_s,voltage,current,temperature,soc_rel
0,549_C20DisCh,0.0,4.17604,-0.15069,23.976151,0.935729
1,549_C20DisCh,60.0,4.17301,-0.15325,23.976151,0.934869
2,549_C20DisCh,120.0,4.17014,-0.15069,23.976151,0.934023


In [5]:
# CÉLULA 6 — Cálculo das derivadas temporais (dV/dt e dI/dt)

dt = dfm.groupby(col_file)[col_time].diff()

mask = dt.isna() | (dt > 0)
dfm = dfm.loc[mask].reset_index(drop=True)

dt = dfm.groupby(col_file)[col_time].diff()

dfm["dv_dt"] = dfm.groupby(col_file)[col_v].diff() / dt
dfm["di_dt"] = dfm.groupby(col_file)[col_i].diff() / dt

dfm.replace([np.inf, -np.inf], np.nan, inplace=True)

print("✔ Derivadas calculadas")
dfm[["dv_dt", "di_dt"]].head()


✔ Derivadas calculadas


,dv_dt,di_dt
0,NaN,NaN
1,-0.000051,-0.000043
2,-0.000048,0.000043
3,-0.000042,0.000000
4,-0.000042,-0.000043


In [6]:
# CÉLULA 7 — Dataset final de features

feature_cols = [col_v, col_i, col_t, "dv_dt", "di_dt"]
target_col = col_soc

dfm = dfm.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

for c in feature_cols + [target_col]:
    dfm[c] = dfm[c].astype("float32")

print("✔ Dataset final:", dfm.shape)


✔ Dataset final: (577186, 8)


### Análise dos Resultados — Dataset Final para Modelos Leves

Após o cálculo das derivadas temporais e remoção de valores inválidos, o conjunto final contém aproximadamente 577 mil amostras. Esta redução reflete a remoção de instantes iniciais sem derivadas válidas, mantendo a qualidade dos dados.

O conjunto final inclui apenas features simples e de baixo custo computacional, adequadas a implementação embarcada.


In [7]:
# CÉLULA 8 — Split treino/teste por ensaio (file_id)

rng = np.random.default_rng(42)

file_ids = dfm[col_file].unique()
rng.shuffle(file_ids)

split = int(0.8 * len(file_ids))
train_ids = set(file_ids[:split])
test_ids  = set(file_ids[split:])

train_mask = dfm[col_file].isin(train_ids)
test_mask  = dfm[col_file].isin(test_ids)

X_train = dfm.loc[train_mask, feature_cols]
y_train = dfm.loc[train_mask, target_col]
X_test  = dfm.loc[test_mask, feature_cols]
y_test  = dfm.loc[test_mask, target_col]

leak = set(dfm.loc[train_mask, col_file]).intersection(
       set(dfm.loc[test_mask, col_file]))

print("✔ Split concluído | leakage file_id:", len(leak))
print("Train:", X_train.shape, "Test:", X_test.shape)


✔ Split concluído | leakage file_id: 0
Train: (477205, 5) Test: (99981, 5)


In [8]:
# CÉLULA 9 — Função de avaliação de regressão

def eval_regression(y_true, y_pred, name):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

results = []


In [9]:
# CÉLULA 10 — Árvore de decisão rasa

from sklearn.tree import DecisionTreeRegressor

for depth in [2, 3, 4, 5, 6]:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append(eval_regression(y_test, pred, f"DT_depth_{depth}"))

pd.DataFrame(results).sort_values("MAE")


,model,MAE,RMSE,R2
4,DT_depth_6,0.111649,0.168883,0.743377
3,DT_depth_5,0.122909,0.181340,0.704126
2,DT_depth_4,0.143514,0.206386,0.616751
1,DT_depth_3,0.165402,0.226856,0.536955
0,DT_depth_2,0.193566,0.274061,0.324202


### Análise dos Resultados — Árvores de Decisão Rasas

As árvores de decisão rasas apresentam desempenho crescente com o aumento da profundidade. A profundidade 6 oferece o melhor compromisso entre erro e complexidade, com MAE ≈ 0,11 e R² ≈ 0,74.

Mesmo com baixa profundidade, o modelo captura relações não lineares relevantes entre tensão, corrente, temperatura e SoC_rel.


In [10]:
# CÉLULA 11 — Regressão linear regularizada (Ridge)

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

for alpha in [0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append(eval_regression(y_test, pred, f"Ridge_{alpha}"))

pd.DataFrame(results).sort_values("MAE").head(10)


,model,MAE,RMSE,R2
4,DT_depth_6,0.111649,0.168883,0.743377
3,DT_depth_5,0.122909,0.181340,0.704126
2,DT_depth_4,0.143514,0.206386,0.616751
1,DT_depth_3,0.165402,0.226856,0.536955
6,Ridge_0.001,0.176070,0.283515,0.276774
5,Ridge_0.0,0.176070,0.283515,0.276774
7,Ridge_0.01,0.176070,0.283515,0.276774
8,Ridge_0.1,0.176071,0.283515,0.276774
9,Ridge_1.0,0.176071,0.283515,0.276775
10,Ridge_10.0,0.176072,0.283513,0.276782


### Análise dos Resultados — Regressão Linear Regularizada

A regressão Ridge apresentou desempenho inferior às árvores de decisão, com MAE elevado e baixo R². A variação do parâmetro de regularização não produziu melhorias significativas.

Este resultado indica que relações puramente lineares são insuficientes para modelar a dinâmica do SoC_rel neste dataset.


In [11]:
# CÉLULA 12 — Comparação final dos modelos leves

df_results = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
df_results


,model,MAE,RMSE,R2
0,DT_depth_6,0.111649,0.168883,0.743377
1,DT_depth_5,0.122909,0.181340,0.704126
2,DT_depth_4,0.143514,0.206386,0.616751
3,DT_depth_3,0.165402,0.226856,0.536955
4,Ridge_0.001,0.176070,0.283515,0.276774
5,Ridge_0.0,0.176070,0.283515,0.276774
6,Ridge_0.01,0.176070,0.283515,0.276774
7,Ridge_0.1,0.176071,0.283515,0.276774
8,Ridge_1.0,0.176071,0.283515,0.276775
9,Ridge_10.0,0.176072,0.283513,0.276782


### Análise dos Resultados — Comparação de Modelos Leves

A comparação global confirma a superioridade das árvores de decisão rasas face à regressão linear. O modelo com profundidade 6 apresenta o melhor compromisso entre desempenho preditivo e simplicidade estrutural.

Este comportamento reforça a adequação de modelos baseados em regras simples para aplicações embarcadas.


In [12]:
# CÉLULA 13 — Seleção do modelo para BMS embarcado

best_model = df_results.loc[0]
print("✔ Modelo selecionado:")
display(best_model)


✔ Modelo selecionado:


model    DT_depth_6
MAE        0.111649
RMSE       0.168883
R2         0.743377
Name: 0, dtype: object

### Análise dos Resultados — Seleção do Modelo Embarcado

Com base nas métricas avaliadas, o modelo DT_depth_6 foi selecionado como o mais adequado para implementação em BMS embarcado. O modelo oferece erro aceitável com complexidade computacional reduzida.

Esta escolha privilegia robustez, interpretabilidade e viabilidade em hardware de baixo consumo.


In [13]:
# ============================================================
# CÉLULA 14 — Treino final e persistência do modelo leve selecionado
# ============================================================

from sklearn.tree import DecisionTreeRegressor
import joblib

# Modelo escolhido com base na comparação de desempenho
best_depth = 6

final_model = DecisionTreeRegressor(
    max_depth=best_depth,
    random_state=42
)

# Treino final com o conjunto de treino completo
final_model.fit(X_train, y_train)

# Guardar modelo treinado para utilização futura (embedded / inferência)
joblib.dump(final_model, "dt_soc_light_depth6.pkl")

print("✔ Modelo final treinado e guardado com sucesso:")
print("  -> dt_soc_light_depth6.pkl")


✔ Modelo final treinado e guardado com sucesso:
  -> dt_soc_light_depth6.pkl


In [14]:
# === CÉLULA 15 — Verificação estrutural do modelo FINAL selecionado ===

# Usar explicitamente o modelo final treinado (Decision Tree depth=6)
estimator = final_model

print("Tipo de modelo:", type(estimator).__name__)

if hasattr(estimator, "tree_"):
    # Caso: Árvore de decisão
    print("Profundidade:", estimator.get_depth())
    print("Número de nós:", estimator.tree_.node_count)

elif hasattr(estimator, "coef_"):
    # Caso: Regressão linear (Ridge)
    print("Número de coeficientes:", estimator.coef_.shape[0])

print("Número de features:", estimator.n_features_in_)

print(
    "\nConclusão: a verificação estrutural analisa o modelo FINAL selecionado, "
    "confirmando complexidade reduzida e compatibilidade com implementação "
    "em microcontroladores de baixo poder computacional (ex.: ESP32)."
)


Tipo de modelo: DecisionTreeRegressor
Profundidade: 6
Número de nós: 127
Número de features: 5

Conclusão: a verificação estrutural analisa o modelo FINAL selecionado, confirmando complexidade reduzida e compatibilidade com implementação em microcontroladores de baixo poder computacional (ex.: ESP32).


## Conclusão — Modelos Leves

Foi identificado um modelo leve com compromisso adequado entre desempenho
preditivo e complexidade computacional.

Este modelo é considerado apropriado para implementação num sistema BMS
embarcado (ex.: ESP32), servindo como alternativa viável aos modelos
computacionalmente mais exigentes (Random Forest e ANN).


In [15]:
# ============================================================
# CÉLULA FINAL — Exportar modelo leve + features para Caminho A
# (aplicar ao CSV real do ESP32 no notebook de validação)
# ============================================================

import os, json, joblib
from datetime import datetime
from sklearn.tree import DecisionTreeRegressor

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# --- 1) Definir/confirmar o modelo final (árvore depth=6) ---
best_depth = 6

final_model = DecisionTreeRegressor(
    max_depth=best_depth,
    random_state=42
)
final_model.fit(X_train, y_train)

# --- 2) Guardar modelo em /kaggle/working (para virar "Output files") ---
model_path = os.path.join(OUT_DIR, f"dt_socrel_depth{best_depth}.joblib")
joblib.dump(final_model, model_path)

# --- 3) Guardar lista de features (ordem importa!) ---
# No teu notebook, feature_cols = [col_v, col_i, col_t, "dv_dt", "di_dt"]
feat_path = os.path.join(OUT_DIR, "feature_cols.json")
with open(feat_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# --- 4) Guardar metadata para ajudar no mapeamento do CSV real ---
schema = {
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "model": f"DecisionTreeRegressor_depth_{best_depth}",
    "target": "soc_rel",
    "feature_cols_in_order": feature_cols,
    "expected_units": {
        "voltage": "V",
        "current": "A",          # no dataset Dhakal, current está em A (descarga tende a ser negativa)
        "temperature": "°C",
        "dv_dt": "V/s",
        "di_dt": "A/s"
    },
    "notes_for_esp32_csv_mapping": {
        "voltage": "usar Vbat_V (ou Vbus_V se não houver Vbat_V)",
        "current": "I_mA / 1000.0 para A; confirmar sinal vs treino",
        "temperature": "T_C",
        "dv_dt": "diff(voltage)/dt",
        "di_dt": "diff(current)/dt"
    }
}
schema_path = os.path.join(OUT_DIR, "model_schema.json")
with open(schema_path, "w") as f:
    json.dump(schema, f, indent=2)

# --- 5) Guardar tabela de resultados (opcional) ---
try:
    df_results.to_csv(os.path.join(OUT_DIR, "metrics_lightweight.csv"), index=False)
except Exception:
    pass

# --- 6) Confirmar outputs ---
print("\n✅ Outputs guardados em /kaggle/working:")
for fn in ["dt_socrel_depth6.joblib", "feature_cols.json", "model_schema.json", "metrics_lightweight.csv"]:
    p = os.path.join(OUT_DIR, fn)
    print(" -", fn, "OK" if os.path.exists(p) else "MISSING")

print("\n📌 Próximo passo no Kaggle:")
print("1) Save Version deste notebook com 'Save output files' ativo")
print("2) No notebook de validação, Add Input do output e carregar:")
print("   /kaggle/input/<pasta>/dt_socrel_depth6.joblib")



✅ Outputs guardados em /kaggle/working:
 - dt_socrel_depth6.joblib OK
 - feature_cols.json OK
 - model_schema.json OK
 - metrics_lightweight.csv OK

📌 Próximo passo no Kaggle:
1) Save Version deste notebook com 'Save output files' ativo
2) No notebook de validação, Add Input do output e carregar:
   /kaggle/input/<pasta>/dt_socrel_depth6.joblib


/tmp/ipykernel_18/1666602525.py:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_utc": datetime.utcnow().isoformat() + "Z",
